# M3: Cosmos Curator — Video Data Curation (NeMo Curator)

**Pipeline Position:** Stage 3: Data Curation — clip splitting, transcoding, motion filtering

| Item | Detail |
|------|--------|
| Tool | **NVIDIA NeMo Curator** v1.2.0 (video pipeline) — the pip-installable sibling of the blog's `cosmos-curate` |
| Input | nuScenes CAM_FRONT frames selected in **M1** (`m1/manifest.json`, tagged by scene) + **M2** captions (`m2/captions.json`) |
| Output | `users/{profile}/m3/` — curated clips manifest + `curated_captions.json` (for M9) |
| Instance | `ml.g5.12xlarge` / `ml.g6.12xlarge` (GPU box for the NVMe scratch; the curation subset runs on CPU) |
| Repo | [NVIDIA-NeMo/Curator](https://github.com/NVIDIA-NeMo/Curator) |

### What this module does

The AWS AV 3.0 blog curates data with **NVIDIA Cosmos Curator** on SageMaker
HyperPod. That tool is Docker/SLURM-only, so this notebook runs the **same
pipeline shape** with its pip-installable sibling, **NeMo Curator**:

1. **Assemble clips** — M1's CAM_FRONT frames are grouped **by scene** and
   stitched into one short `.mp4` per scene with `cv2.VideoWriter` at a **fixed
   FPS**. nuScenes frames arrive at mixed rates (2 Hz keyframes, ~12 Hz sweeps);
   writing at a fixed FPS **normalizes them to a uniform playback rate**. *(This
   FPS normalization is an ffmpeg/OpenCV step — NeMo Curator reads framerate as
   metadata but does not itself re-encode to a target FPS.)*
2. **Real NeMo Curator video pipeline** — `VideoReader → FixedStride split →
   transcode (H.264) → motion filter → manifest`. The motion filter
   **actually removes low-motion clips** (e.g. the ego vehicle stopped at a
   light), which is where curation earns its keep.
3. **Preserve the pipeline contract** — M9 trains on `m3/curated_captions.json`.
   We keep writing M2's captions there, annotated with the curation verdict of
   the clip each caption's scene produced, so the end-to-end pipeline stays real.

> **Environment:** installed by `scripts/setup_nemo_curator_env.sh` into a
> dedicated `uv` venv on the instance NVMe (isolated from the notebook kernel).
> The first cell runs it. Ephemeral — re-run after an app restart. No HF token
> and no model downloads are needed for this subset (the motion filter is
> model-free).

In [ ]:
# ============================================================
# Setup — configuration
# ============================================================
import os
import sys
import time
import json
import glob
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import boto3

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PROFILE = os.environ.get("USER_PROFILE", os.environ.get("BLUEPRINT_PROFILE", "default"))
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")

# Inputs: M1 frame manifest (with per-frame scene tags) + M2 captions.
M1_PREFIX = f"users/{PROFILE}/m1/"
M2_PREFIX = f"users/{PROFILE}/m2/"
OUTPUT_PREFIX = f"users/{PROFILE}/m3/"
NUSCENES_PREFIX = "datasets/nuscenes-mini/"   # in SHARED_BUCKET

# Local scratch on the instance NVMe.
NVME = "/mnt/sagemaker-nvme" if os.path.isdir("/mnt/sagemaker-nvme") else "/tmp"
WORK = f"{NVME}/m3_work"
CLIPS_IN_DIR = f"{WORK}/clips_in"       # per-scene mp4s we assemble (NeMo Curator input)
CURATE_OUT_DIR = f"{WORK}/curated"      # NeMo Curator output (clips/, filtered_clips/, metas/)
for d in (WORK, CLIPS_IN_DIR, CURATE_OUT_DIR):
    os.makedirs(d, exist_ok=True)

# NeMo Curator env produced by scripts/setup_nemo_curator_env.sh
NC_WORK = f"{NVME}/nemo-curator-work"
NC_ENV_FILE = f"{NC_WORK}/nemo_curator_env.sh"
NC_VENV_PY = f"{NC_WORK}/.venv/bin/python"

# Clip assembly params. VIDEO_FPS is the uniform rate we normalize every scene
# to. clip_len_s * VIDEO_FPS should be <= frames per scene (~40 keyframes).
VIDEO_FPS = 10
VIDEO_W, VIDEO_H = 1280, 704
CLIP_LEN_S = 2.0        # fixed_stride split length inside NeMo Curator

s3 = boto3.client("s3")
print(f"Account ID: {ACCOUNT_ID}")
print(f"Profile:    {PROFILE}")
print(f"Input:      s3://{USER_BUCKET}/{M1_PREFIX} (+ {M2_PREFIX})")
print(f"Output:     s3://{USER_BUCKET}/{OUTPUT_PREFIX}")
print(f"NVMe work:  {WORK}")
print("\nSetup complete.")

In [ ]:
# ============================================================
# Install / activate the NeMo Curator environment
# ============================================================
# Runs scripts/setup_nemo_curator_env.sh (local copy, else the S3-staged one).
# It creates a dedicated uv venv on the NVMe with nemo-curator[video_cpu]==1.2.0
# and ensures a working ffmpeg with H.264 + VP9 encoders. Idempotent; ephemeral (re-run
# after an app restart). 5-15 min on first run.

def _find_setup_script():
    for base in [Path.cwd(), Path.cwd().parent, Path.home()]:
        cand = base / "scripts" / "setup_nemo_curator_env.sh"
        try:
            if cand.exists():
                return str(cand)
        except OSError:
            continue
    return None

setup_script = _find_setup_script()
if setup_script is None:
    local = f"{WORK}/setup_nemo_curator_env.sh"
    subprocess.run(
        ["aws", "s3", "cp",
         f"s3://{SHARED_BUCKET}/notebook-templates/scripts/setup_nemo_curator_env.sh", local],
        check=True,
    )
    setup_script = local

print(f"Running setup script: {setup_script}")
print("(first run: 5-15 min for the uv venv + nemo-curator install; re-runs are fast)\n")
proc = subprocess.run(["bash", setup_script], text=True)
if proc.returncode != 0:
    raise RuntimeError(
        "setup_nemo_curator_env.sh failed — scroll up for the error. Common causes: "
        "not on a GPU instance (no /mnt/sagemaker-nvme), no network egress to PyPI, "
        "or an ffmpeg without the libopenh264 encoder."
    )

# Confirm the venv python exists and the video-pipeline imports resolve.
assert os.path.exists(NC_VENV_PY), f"curator venv python not found at {NC_VENV_PY}"
check = subprocess.run(
    [NC_VENV_PY, "-c",
     "from nemo_curator.pipeline import Pipeline; "
     "from nemo_curator.stages.video.clipping.clip_extraction_stages import "
     "FixedStrideExtractorStage, ClipTranscodingStage; print('nemo-curator import OK')"],
    text=True, capture_output=True,
)
print(check.stdout.strip() or check.stderr.strip())
if check.returncode != 0:
    raise RuntimeError("NeMo Curator video imports failed — see setup output above.")
print("\nNeMo Curator environment ready.")

In [ ]:
# ============================================================
# Pre-flight — GPU box check
# ============================================================
# This module needs a GPU *instance* for its large NVMe scratch disk, but the
# split/transcode/motion-filter subset itself runs on CPU (motion scoring uses
# --motion-score-gpus-per-worker 0). So we check for the NVMe, and report the
# GPU if present, without hard-failing on VRAM.
import shutil as _sh

if NVME == "/tmp":
    print("WARNING: /mnt/sagemaker-nvme not found — using /tmp. Fine for a small\n"
          "         test, but a GPU instance gives the large NVMe scratch this wants.")
free_gb = _sh.disk_usage(WORK).free / (1024**3)
print(f"Scratch dir: {WORK}  ({free_gb:.0f} GB free)")

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True, capture_output=True,
    )
    if out.returncode == 0 and out.stdout.strip():
        print("GPU(s):")
        for line in out.stdout.strip().splitlines():
            print(f"  {line}")
    else:
        print("No GPU detected (fine — the curation subset runs on CPU).")
except FileNotFoundError:
    print("nvidia-smi not found (fine — the curation subset runs on CPU).")

# ffmpeg with the libopenh264 encoder must be present (setup cell installs it).
# H.264 is required so the motion filter can read motion-vector side data
# (ffmpeg's VP9 decoder does not export it). Verify ffmpeg RUNS.
_ff = subprocess.run(["bash", "-lc",
    "ffmpeg -hide_banner -version >/dev/null 2>&1 && "
    "ffmpeg -hide_banner -encoders 2>/dev/null | grep -q libopenh264 && echo OK"],
    text=True, capture_output=True)
assert _ff.stdout.strip() == "OK", (
    "ffmpeg with libopenh264 not available — re-run the setup cell (it force-"
    "reinstalls a modern conda-forge ffmpeg)."
)
print("ffmpeg + libopenh264: OK")
print("Pre-flight PASSED.")

In [ ]:
# ============================================================
# Assemble one mp4 per scene from M1's CAM_FRONT frames (Hz normalization)
# ============================================================
# M1's manifest lists CAM_FRONT frames and, per frame, the scene it came from
# (cam_front_scenes). We group frames by scene and write one mp4 per scene at a
# FIXED VIDEO_FPS. nuScenes frames arrive at mixed rates; writing at a fixed FPS
# normalizes every scene to a uniform playback rate — the honest "unify video
# Hz" step. (This is cv2/ffmpeg, not a NeMo Curator feature.)
#
# cv2 is not in the SMD kernel; install the headless build with --no-deps so it
# can't bump numpy and break other SMD packages.
try:
    import cv2
except ModuleNotFoundError:
    print("Installing opencv-python-headless (--no-deps) into the kernel...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "opencv-python-headless"], check=True)
    import cv2
print(f"cv2 {cv2.__version__} ready")

# 1) Read M1's manifest (frames + per-frame scene tags).
m1_key = f"{M1_PREFIX}manifest.json"
m1 = json.loads(s3.get_object(Bucket=USER_BUCKET, Key=m1_key)["Body"].read())
cam_files = m1["cam_front_files"]
cam_scenes = m1.get("cam_front_scenes")
assert cam_files, "M1 manifest has no cam_front_files — re-run M1."
if not cam_scenes or len(cam_scenes) != len(cam_files):
    # Older M1 without scene tags: treat everything as one scene.
    print("NOTE: M1 manifest has no cam_front_scenes tags — assembling a single clip. "
          "Re-run the latest M1 for per-scene curation.")
    cam_scenes = ["scene-000"] * len(cam_files)

# 2) Group frame keys by scene, preserving order.
from collections import OrderedDict
by_scene = OrderedDict()
for f, sc in zip(cam_files, cam_scenes):
    by_scene.setdefault(sc, []).append(f)
print(f"M1 provided {len(cam_files)} frames across {len(by_scene)} scenes")

# 3) Download frames + assemble one mp4 per scene at the uniform VIDEO_FPS.
#    Clear any stale clips from a previous run first.
for old in glob.glob(f"{CLIPS_IN_DIR}/*.mp4"):
    os.remove(old)
frames_cache = f"{WORK}/frames"
os.makedirs(frames_cache, exist_ok=True)

assembled = []
for scene, keys in by_scene.items():
    out_mp4 = f"{CLIPS_IN_DIR}/{scene}.mp4"
    vw = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"),
                         VIDEO_FPS, (VIDEO_W, VIDEO_H))
    n = 0
    for rel in keys:
        dest = os.path.join(frames_cache, os.path.basename(rel))
        if not os.path.exists(dest):
            s3.download_file(SHARED_BUCKET, f"{NUSCENES_PREFIX}{rel}", dest)
        img = cv2.imread(dest)
        if img is None:
            continue
        vw.write(cv2.resize(img, (VIDEO_W, VIDEO_H)))
        n += 1
    vw.release()
    if n > 0:
        dur = n / VIDEO_FPS
        assembled.append({"scene": scene, "mp4": out_mp4, "frames": n, "duration_s": round(dur, 1)})
        print(f"  {scene}: {n} frames -> {os.path.basename(out_mp4)} ({dur:.1f}s @ {VIDEO_FPS}fps)")

assert assembled, "No clips assembled — check M1 frames staged in the shared bucket."
print(f"\nAssembled {len(assembled)} scene clips (normalized to {VIDEO_FPS} fps) in {CLIPS_IN_DIR}")

In [ ]:
# ============================================================
# Run the REAL NeMo Curator video pipeline (split -> transcode -> motion filter)
# ============================================================
# Shell out to the curator venv's shipped example. CPU H.264 encoder and
# CPU motion scoring (--motion-score-gpus-per-worker 0) — model-free, so no HF
# token / no downloads. Embeddings + captions are disabled.
#
# --- Motion filter thresholds (tunable — this is the curation knob) ---------
# The motion filter KEEPS a clip iff:
#     global_mean >= MOTION_GLOBAL_MEAN_THRESHOLD  AND
#     per_patch_min_256 >= MOTION_PER_PATCH_THRESHOLD
# global_mean is the average motion across the clip. per_patch_min_256 is the
# motion of the LEAST-moving coarse patch — which is ~0 for almost any real
# driving frame (sky / distant road / background are always near-static). So
# NeMo Curator's DEFAULT per-patch threshold (1e-6) effectively requires EVERY
# region to move and drops virtually everything. We set it to 0 to disable that
# gate and curate on global_mean alone — the meaningful signal here. Raise
# MOTION_GLOBAL_MEAN_THRESHOLD to keep fewer (more-dynamic) clips, lower it to
# keep more. On nuScenes keyframe clips, scores run ~0.0004-0.004 (median
# ~0.0012), so the default below keeps the more dynamic ~half.
MOTION_GLOBAL_MEAN_THRESHOLD = 0.0012   # ~median of this data; tune to taste
MOTION_PER_PATCH_THRESHOLD = 0.0        # 0 disables the "every patch must move" gate
# ---------------------------------------------------------------------------
#
# Clear any previous output so counts are fresh.
if os.path.isdir(CURATE_OUT_DIR):
    shutil.rmtree(CURATE_OUT_DIR)
os.makedirs(CURATE_OUT_DIR, exist_ok=True)

# Resolve the example script path from the env file the setup script wrote.
example = os.environ.get("NEMO_CURATOR_EXAMPLE", "")
if not example or not os.path.exists(example):
    # parse it out of the env file
    for line in Path(NC_ENV_FILE).read_text().splitlines():
        if line.startswith("export NEMO_CURATOR_EXAMPLE="):
            example = line.split("=", 1)[1].strip().strip('"')
assert example and os.path.exists(example), (
    f"video_split_clip_example.py not found ({example}). Re-run the setup cell.")

cmd = [
    NC_VENV_PY, example,
    "--video-dir", CLIPS_IN_DIR,
    "--output-path", CURATE_OUT_DIR,
    "--splitting-algorithm", "fixed_stride",
    "--fixed-stride-split-duration", str(CLIP_LEN_S),
    "--fixed-stride-min-clip-length-s", "1.0",
    # Software H.264 (CPU). Required for the motion filter: it reads motion-vector
    # side data, which ffmpeg exports for H.264/MPEG but NOT for VP9 (a VP9 clip
    # yields motion_score -1 and is dropped as "no_motion_frames").
    "--transcode-encoder", "libopenh264",
    "--motion-filter", "enable",           # drop low-motion clips (curate on motion)
    "--motion-score-gpus-per-worker", "0", # CPU motion scoring
    "--motion-global-mean-threshold", str(MOTION_GLOBAL_MEAN_THRESHOLD),
    "--motion-per-patch-min-256-threshold", str(MOTION_PER_PATCH_THRESHOLD),
    "--no-generate-embeddings",            # model-free subset
    "--verbose",
]
print("Running NeMo Curator:\n  " + " ".join(cmd) + "\n")
t0 = time.time()
proc = subprocess.run(cmd, text=True, capture_output=True,
                      env={**os.environ, "LOGURU_LEVEL": "INFO"})
elapsed = time.time() - t0
# Show the tail of the run log (it's verbose).
tail = "\n".join((proc.stdout + proc.stderr).splitlines()[-25:])
print(tail)
print(f"\nNeMo Curator finished in {elapsed:.1f}s (exit {proc.returncode})")
if proc.returncode != 0:
    raise RuntimeError("NeMo Curator pipeline failed — see the log tail above.")

In [ ]:
# ============================================================
# Curation results — what NeMo Curator kept vs filtered
# ============================================================
import re
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

def _list(sub):
    return sorted(glob.glob(f"{CURATE_OUT_DIR}/{sub}/**/*", recursive=True))

kept_clips = [p for p in _list("clips") if p.endswith(".mp4")]
filtered_clips = [p for p in _list("filtered_clips") if p.endswith(".mp4")]
meta_files = sorted(glob.glob(f"{CURATE_OUT_DIR}/metas/**/*.json", recursive=True))

# Load per-clip metadata. NeMo Curator names clips by UUID, but each meta records
# source_video (".../<scene>.mp4") and clip_location (".../clips/" if kept, else
# ".../filtered_clips/"). Use those to map each clip back to its scene AND its
# verdict — the UUID filenames alone can't be mapped to scenes.
metas = []
for m in meta_files:
    try:
        metas.append(json.loads(Path(m).read_text()))
    except Exception:
        pass

def _scene_of(meta):
    src = meta.get("source_video", "")
    return Path(src).stem if src else None      # ".../scene-0553.mp4" -> "scene-0553"

def _verdict_of(meta):
    loc = meta.get("clip_location", "")
    if "/filtered_clips/" in loc:
        return "filtered"
    if "/clips/" in loc:
        return "kept"
    return "kept" if meta.get("valid") else "filtered"

# Per-scene tally: how many of each scene's split-clips were kept vs filtered.
from collections import defaultdict
scene_tally = defaultdict(lambda: {"kept": 0, "filtered": 0})
for md in metas:
    sc = _scene_of(md)
    if sc:
        scene_tally[sc][_verdict_of(md)] += 1

# A scene "survives" curation if at least one of its clips passed the motion filter.
scenes_kept = sorted(sc for sc, t in scene_tally.items() if t["kept"] > 0)

n_in = len(assembled)
n_kept = len(kept_clips)
n_filtered = len(filtered_clips)
n_clips_total = n_kept + n_filtered
print("=" * 60)
print("NeMo Curator — Video Curation Results")
print("=" * 60)
print(f"  Input scene clips:        {n_in}")
print(f"  Clips after split:        {n_clips_total}  (fixed_stride @ {CLIP_LEN_S}s)")
print(f"  Kept (passed motion):     {n_kept}")
print(f"  Filtered (low motion):    {n_filtered}")
if n_clips_total:
    print(f"  Retention rate:           {n_kept/n_clips_total*100:.1f}%")
print(f"  Scenes with >=1 kept clip: {len(scenes_kept)}/{len(scene_tally)}")
print("\n  Per-scene (kept/filtered clips):")
for sc in sorted(scene_tally):
    t = scene_tally[sc]
    print(f"    {sc}: {t['kept']} kept / {t['filtered']} filtered")

# Motion-score distribution (present when the motion filter ran).
motion_scores = []
for md in metas:
    ms = md.get("motion_score")
    if isinstance(ms, dict) and "global_mean" in ms:
        motion_scores.append(ms["global_mean"])
    elif isinstance(ms, (int, float)):
        motion_scores.append(ms)

if motion_scores:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(motion_scores, bins=20, color="steelblue", alpha=0.8)
    ax.axvline(MOTION_GLOBAL_MEAN_THRESHOLD, color="red", linestyle="--",
               label=f"keep threshold ({MOTION_GLOBAL_MEAN_THRESHOLD})")
    ax.set_xlabel("Motion score (global mean)")
    ax.set_ylabel("Clip count")
    ax.set_title(f"NeMo Curator motion scores across {len(motion_scores)} clips")
    ax.legend()
    plt.tight_layout()
    plt.savefig("/tmp/m3_motion_scores.png", dpi=100, bbox_inches="tight")
    plt.show()
    display(IPImage(filename="/tmp/m3_motion_scores.png"))
    print(f"\nMotion score range: {min(motion_scores):.6f} - {max(motion_scores):.6f}")
else:
    print("\n(no motion_score in metadata — the motion filter may not have run; "
          "check the log tail in the previous cell)")

In [ ]:
# ============================================================
# Write curated outputs to S3 (preserve the M9 training contract)
# ============================================================
# M9 trains on m3/curated_captions.json -> {"curated_captions": [...]}. We keep
# passing M2's captions through so the end-to-end pipeline stays real, annotating
# each caption with the curation verdict for its scene's clip. Separately we
# write m3/curation_report.json with the video-curation stats.
now = datetime.now(timezone.utc).isoformat()

# Which scenes survived motion filtering? cell 6 computed scenes_kept from the
# clip metadata (source_video -> scene, clip_location -> kept/filtered), which is
# the only reliable mapping (NeMo Curator names clips by UUID). Use it directly.
kept_scenes = set(scenes_kept)

# Load M2 captions and annotate with the curation verdict.
try:
    m2 = json.loads(s3.get_object(Bucket=USER_BUCKET, Key=f"{M2_PREFIX}captions.json")["Body"].read())
    m2_caps = m2.get("captions", [])
except Exception as e:
    print(f"NOTE: could not read M2 captions ({e}); writing an empty caption list.")
    m2_caps = []

# Annotate each M2 caption with the curation verdict of the scene it came from.
# If the mapping came back empty (unexpected), fall back to keeping all so M9
# still gets training data rather than an empty set.
_have_mapping = len(kept_scenes) > 0
curated_captions = []
for c in m2_caps:
    sc = c.get("scene")
    if _have_mapping:
        verdict = "kept" if sc in kept_scenes else "filtered"
    else:
        verdict = "kept"
    curated_captions.append({**c, "curation_verdict": verdict})

# M9 trains on captions whose scene survived curation. Fall back to all captions
# only if we somehow couldn't map any scene (keeps M9 fed).
kept_caption_list = [c for c in curated_captions if c["curation_verdict"] == "kept"] or curated_captions

curated_output = {
    "module": "M3_Cosmos_Curator",
    "generated_at": now,
    "curator": "nemo-curator/video (split+transcode+motion-filter)",
    "source_modules": ["M1_Data_Exploration", "M2_Cosmos_Reason_Captioning"],
    "curation_stats": {
        "input_scene_clips": len(assembled),
        "clips_after_split": len(kept_clips) + len(filtered_clips),
        "clips_kept": len(kept_clips),
        "clips_filtered_low_motion": len(filtered_clips),
        "scenes_kept": sorted(kept_scenes),
        "video_fps_normalized_to": VIDEO_FPS,
        "clip_len_s": CLIP_LEN_S,
    },
    # M9 contract: a flat list under "curated_captions".
    "curated_captions": kept_caption_list,
}
out_key = f"{OUTPUT_PREFIX}curated_captions.json"
s3.put_object(Bucket=USER_BUCKET, Key=out_key,
              Body=json.dumps(curated_output, indent=2, ensure_ascii=False),
              ContentType="application/json")
print(f"Curated data written: s3://{USER_BUCKET}/{out_key}")
print(f"  curated_captions: {len(kept_caption_list)} (M9 trains on these)")

# Video-curation report.
report = {
    "module": "M3_Cosmos_Curator",
    "timestamp": now,
    "tool": "NVIDIA NeMo Curator v1.2.0 (video pipeline)",
    "input_path": f"s3://{USER_BUCKET}/{M1_PREFIX}manifest.json",
    "output_path": f"s3://{USER_BUCKET}/{out_key}",
    "stats": curated_output["curation_stats"],
}
report_key = f"{OUTPUT_PREFIX}curation_report.json"
s3.put_object(Bucket=USER_BUCKET, Key=report_key,
              Body=json.dumps(report, indent=2), ContentType="application/json")
print(f"Curation report written: s3://{USER_BUCKET}/{report_key}")

# Upload the kept clips too (small H.264 mp4s), for inspection / M4.
uploaded = 0
for p in kept_clips:
    s3.upload_file(p, USER_BUCKET, f"{OUTPUT_PREFIX}clips/{os.path.basename(p)}")
    uploaded += 1
print(f"Uploaded {uploaded} curated clip(s) to s3://{USER_BUCKET}/{OUTPUT_PREFIX}clips/")
print(f"\nNext: M9 reads {out_key}; M4 augments the M1 frames independently.")

In [ ]:
# ============================================================
# Cost Analysis
# ============================================================
# Read the actual instance type from the SageMaker resource metadata when
# available, so the estimate reflects the box you're really on.
INSTANCE_RATES = {
    "ml.t3.medium": 0.056, "ml.g5.12xlarge": 7.09, "ml.g6.12xlarge": 6.69,
    "ml.p4d.24xlarge": 37.69, "ml.p5.48xlarge": 113.14,
}
inst = "ml.g6.12xlarge"
try:
    md = json.loads(Path("/opt/ml/metadata/resource-metadata.json").read_text())
    inst = md.get("InstanceType", inst)
except Exception:
    pass
rate = INSTANCE_RATES.get(inst, 6.69)
KRW = 1370

# curation wall-clock came from cell 5's `elapsed` if still in scope; else estimate.
try:
    curate_s = float(elapsed)
except NameError:
    curate_s = 60.0
est_min = max(curate_s / 60.0, 1.0) + 10  # + setup/assembly overhead
cost = rate * (est_min / 60.0)

print("=" * 60)
print("M3 NeMo Curator — Cost Analysis")
print("=" * 60)
print(f"Instance:            {inst}  (${rate:.2f}/hr)")
print(f"Curation wall-clock: {curate_s:.0f}s  (CPU split/transcode/motion)")
print(f"Est. module time:    {est_min:.0f} min (incl. setup + clip assembly)")
print(f"Compute cost:        ${cost:.2f} USD ({cost*KRW:.0f} KRW)")
print(f"S3 operations:       ~$0.01 USD (negligible)")
print("=" * 60)
print("Note: the curation subset (split/transcode/motion) is CPU-bound and")
print("model-free. The GPU box is used for its NVMe scratch; captioning and")
print("embedding stages (which need the GPU) are intentionally disabled here.")

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m03-cosmos-curator")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")